# Data Collection

This notebook downloads:
1. Cryptocurrency market data (BTC, ETH, BNB, ADA, SOL, DOGE) from Binance
2. Fear & Greed sentiment index (value)
3. FRED macroeconomic indicators

Data period: 2018-05-01 to 2025-01-01
Output: CSV files in data/ folder

Note: Binance only have data starting from 2018-04-17

## Cryptocurrency Market Data
OHLCV(BTC, ETH, BNB, ADA, SOL, DOGE)

In [ ]:
import ccxt
import pandas as pd
import requests
import time
import os
from dotenv import load_dotenv

load_dotenv()

SYMBOLS = ['BTC/USDT', 'ETH/USDT', 'BNB/USDT', 'ADA/USDT', 'SOL/USDT', 'DOGE/USDT']
START_DATE = '2018-05-01'
END_DATE = '2025-01-01'
FRED_API_KEY = os.getenv('FRED_API_KEY')

exchange = ccxt.binance({'enableRateLimit': True})

def download_crypto(symbol, timeframe):
    since = exchange.parse8601(f'{START_DATE}T00:00:00Z')
    end = exchange.parse8601(f'{END_DATE}T23:59:59Z')
    data = []
    
    print(f"Downloading {symbol} ({timeframe})...")
    
    while since < end:
        ohlcv = exchange.fetch_ohlcv(symbol, timeframe, since, limit=1000)
        if not ohlcv:
            break
        data.extend(ohlcv)
        since = ohlcv[-1][0] + 1
        if len(ohlcv) < 1000:
            break
        time.sleep(exchange.rateLimit / 1000)
    
    df = pd.DataFrame(data, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
    df = df[(df['timestamp'] >= START_DATE) & (df['timestamp'] <= END_DATE)]
    df['symbol'] = symbol.replace('/USDT', '')
    
    return df

daily_data = [download_crypto(symbol, '1d') for symbol in SYMBOLS]
crypto_daily_df = pd.concat(daily_data, ignore_index=True).sort_values(['symbol', 'timestamp'])
crypto_daily_df.to_csv('data/crypto_market_data_daily.csv', index=False)

print(f"Saved crypto_market_data_daily.csv: {len(crypto_daily_df):,} rows")

# hourly_data = [download_crypto(symbol, '1h') for symbol in SYMBOLS]
# crypto_hourly_df = pd.concat(hourly_data, ignore_index=True).sort_values(['symbol', 'timestamp'])
# crypto_hourly_df.to_csv('data/crypto_market_data_hourly.csv', index=False)

# print(f"Saved crypto_market_data_hourly.csv: {len(crypto_hourly_df):,} rows")

## Sentiment Data
CoinMarketCap Crypto Fear and Greed Index
- Metrics for crypto market trader sentiments (e.g. extreme fear, neutral, greed)

In [7]:
response = requests.get("https://api.alternative.me/fng/?limit=0&format=json")
fg = pd.DataFrame(response.json()['data'])
fg['date'] = pd.to_datetime(fg['timestamp'].astype(int), unit='s')
fg['value'] = pd.to_numeric(fg['value'])
fg = fg[(fg['date'] >= START_DATE) & (fg['date'] <= END_DATE)]
fg = fg[['date', 'value', 'value_classification']].sort_values('date').reset_index(drop=True)
fg.to_csv('data/sentiment_data.csv', index=False)
print(f"Saved sentiment_data.csv: {len(fg):,} rows\n")

Saved sentiment_data.csv: 2,437 rows



## Macroeconomic Data
FRED (Federal Reserve Economic Data)
- Federal Funds Effective Rate, 10 Year Treasury Yield

In [ ]:
from fredapi import Fred
import pandas as pd

fred = Fred(api_key=FRED_API_KEY)

macro = pd.DataFrame({
    'fed_funds': fred.get_series('DFF', observation_start=START_DATE, observation_end=END_DATE),
    'treasury_10y': fred.get_series('DGS10', observation_start=START_DATE, observation_end=END_DATE),
    'vix': fred.get_series('VIXCLS', observation_start=START_DATE, observation_end=END_DATE),
    'm2': fred.get_series('M2SL', observation_start=START_DATE, observation_end=END_DATE),
    'cpi': fred.get_series('CPIAUCSL', observation_start=START_DATE, observation_end=END_DATE),
    'real_rate_10y': fred.get_series('DFII10', observation_start=START_DATE, observation_end=END_DATE)
})

macro.index.name = 'date'
macro.to_csv('data/macro_data.csv')
print(f"Saved macro_data.csv: {len(macro)} rows")


## GOOGLE Trends API

In [ ]:
from pytrends.request import TrendReq
import pandas as pd
from datetime import timedelta
import time

pytrends = TrendReq(hl='en-US', tz=360)

keywords = ["BTC", "ETH", "BNB", "ADA", "SOL", "DOGE"]

# split into chunks of 5 (Google limit)
groups = [keywords[i:i+5] for i in range(0, len(keywords), 5)]

start, end = pd.Timestamp("2018-05-01"), pd.Timestamp("2025-01-01")
all_df = []

def fetch_with_retry(kw, tf, max_retries=5):
    wait = 60
    for attempt in range(max_retries):
        try:
            pytrends.build_payload(kw, timeframe=tf)
            df = pytrends.interest_over_time().drop(columns="isPartial", errors="ignore")
            return df
        except Exception as e:
            print(f"  [{attempt+1}/{max_retries}] Error at {tf}: {e} — retrying in {wait}s")
            time.sleep(wait)
            wait *= 2
    print(f"  Giving up on {tf}")
    return None

for g in groups:
    cur, tmp = start, []
    while cur < end:
        nxt = min(cur + timedelta(days=269), end)
        tf = f"{cur.date()} {nxt.date()}"
        df = fetch_with_retry(g, tf)
        if df is not None and not df.empty:
            tmp.append(df)
        time.sleep(10)
        cur = nxt + timedelta(days=1)

    if tmp:
        df = pd.concat(tmp).loc[lambda x: ~x.index.duplicated()]
        all_df.append(df)

coin_trends = pd.concat(all_df, axis=1).loc[:, lambda x: ~x.columns.duplicated()]
coin_trends.columns = [f"{col}_google_trend" for col in coin_trends.columns]
coin_trends.to_csv("data/google_trends_coin.csv")
print(f"Saved google_trends_coin.csv: {len(coin_trends):,} rows")

In [ ]:
from pytrends.request import TrendReq
from datetime import timedelta
import time

pytrends = TrendReq(hl='en-US', tz=360)

keywords = ['buy bitcoin', 'crypto crash', 'bitcoin price', 'altcoin season', 'sell crypto']

start, end = pd.Timestamp(START_DATE), pd.Timestamp(END_DATE)
tmp = []

def fetch_with_retry(kw, tf, max_retries=5):
    wait = 60
    for attempt in range(max_retries):
        try:
            pytrends.build_payload(kw, timeframe=tf)
            df = pytrends.interest_over_time().drop(columns='isPartial', errors='ignore')
            return df
        except Exception as e:
            print(f"  [{attempt+1}/{max_retries}] Error at {tf}: {e} — retrying in {wait}s")
            time.sleep(wait)
            wait *= 2
    print(f"  Giving up on {tf}")
    return None

cur = start
while cur < end:
    nxt = min(cur + timedelta(days=269), end)
    tf = f"{cur.date()} {nxt.date()}"
    df = fetch_with_retry(keywords, tf)
    if df is not None and not df.empty:
        tmp.append(df)
    time.sleep(10)
    cur = nxt + timedelta(days=1)

market_trends = pd.concat(tmp).loc[lambda x: ~x.index.duplicated()]
market_trends.columns = [col.replace(' ', '_') + '_trend' for col in market_trends.columns]
market_trends.index.name = 'date'
market_trends = market_trends.reset_index()
market_trends.to_csv('data/google_trends_market.csv', index=False)
print(f"Saved google_trends_market.csv: {len(market_trends):,} rows")